<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/T32026_DLP_Week_1_Session.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Setup

In [6]:
!pip install -q -U datasets huggingface_hub fsspec aiohttp aiofiles
import datasets
print("datasets version:", datasets.__version__)

datasets version: 5.1.0


In [7]:
# !pip install -q -U "datasets==3.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.


---
## Part 1 — Loading Datasets from the Hub


**Load IMDb and inspect structure**

In [10]:
from datasets import load_dataset
from pprint import pprint
imdb = load_dataset("stanfordnlp/imdb")
pprint(imdb)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [11]:
pprint(imdb['train'])

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


In [12]:
df = imdb.pop('unsupervised')
pprint(df)

Dataset({
    features: ['text', 'label'],
    num_rows: 50000
})


---
## Part 2 — Local Files, Configs, and Indexing


**Explore a dataset with multiple configs (WMT14)**

Some Hub entries are *collections* of datasets. `wmt14` bundles many language pairs — you must
pick a config before loading.

In [13]:
from datasets import get_dataset_config_names, get_dataset_split_names

In [14]:
configs = get_dataset_config_names("wmt/wmt14")
configs

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

['cs-en', 'de-en', 'fr-en', 'hi-en', 'ru-en']

In [15]:
splits = get_dataset_split_names("wmt/wmt14", "hi-en")
splits

['train', 'validation', 'test']

---
## Part 3 — Filter, Map, and Combine


**Filtering noisy/short samples**

`filter()` runs a boolean function on every row and keeps only the ones that return `True`.
On a `DatasetDict`, it applies across every split automatically.

In [19]:
def is_long_enough(example):
  return len(example['text'])<100

In [20]:
imdb.filter(is_long_enough)

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 11
    })
})

In [24]:
def add_prefix(example):
  return {"text" : "Movie Review: " + example['text']}

In [29]:
ds = imdb.map(add_prefix)
pprint(ds['train'][0]['text'][:1000])

('Movie Review: I rented I AM CURIOUS-YELLOW from my video store because of '
 'all the controversy that surrounded it when it was first released in 1967. I '
 'also heard that at first it was seized by U.S. customs if it ever tried to '
 'enter this country, therefore being a fan of films considered '
 '"controversial" I really had to see this for myself.<br /><br />The plot is '
 'centered around a young Swedish drama student named Lena who wants to learn '
 'everything she can about life. In particular she wants to focus her '
 'attentions to making some sort of documentary on what the average Swede '
 'thought about certain political issues such as the Vietnam War and race '
 'issues in the United States. In between asking politicians and ordinary '
 'denizens of Stockholm about their opinions on politics, she has sex with her '
 'drama teacher, classmates, and married men.<br /><br />What kills me about I '
 'AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic

---
## Part 4 — Streaming Large Datasets

 **Streaming mode**

`streaming=True` returns an `IterableDataset` — no full download, fetches one example at a time.
`map`/`filter` still work, but lazily (applied as each sample is pulled).

In [ ]:
streamed = load_dataset("stanfordnlp/imdb", )

---
## Part 5 — Save/Load Optimized Format

**PyArrow save/load round-trip**